In [3]:
print(df.columns)

Index(['Book Name', 'Author', 'Rating_x', 'Number of Reviews_x', 'Price_x',
       'Rating_y', 'Number of Reviews_y', 'Price_y', 'Description',
       'Listening Time', 'Ranks and Genre'],
      dtype='object')


In [29]:
print(df.head)

<bound method NDFrame.head of                                               Book Name                Author  \
0     Think Like a Monk: The Secret of How to Harnes...            Jay Shetty   
1     Ikigai: The Japanese Secret to a Long and Happ...         Héctor García   
2     The Subtle Art of Not Giving a F*ck: A Counter...           Mark Manson   
3     Atomic Habits: An Easy and Proven Way to Build...           James Clear   
4     Life's Amazing Secrets: How to Find Balance an...        Gaur Gopal Das   
...                                                 ...                   ...   
4291  Factfulness: Wie wir lernen, die Welt so zu se...          Hans Rosling   
4292       Late-Talking Children: A Symptom or a Stage?   Stephen M. Camarata   
4293  The Marketing of Evil: How Radicals, Elitists ...        David Kupelian   
4294      Things I Wish I'd Known Before We Got Married          Gary Chapman   
4295  The Disease Delusion: Conquering the Causes of...  Dr. Jeffrey S. Bland  

In [ ]:
def clean_data(df):
    df = df.drop_duplicates()

    # Rating
    df['Rating'] = pd.to_numeric(df['Rating'], errors='coerce')
    df['Rating'] = df['Rating'].fillna(df['Rating'].mean())

    # Reviews
    df['Number of Reviews'] = pd.to_numeric(df['Number of Reviews'], errors='coerce')
    df['Number of Reviews'] = df['Number of Reviews'].fillna(0)

    # Price
    df['Price'] = df['Price'].astype(str).str.replace('[₹,]', '', regex=True)
    df['Price'] = pd.to_numeric(df['Price'], errors='coerce')
    df['Price'] = df['Price'].fillna(0)

    # Text columns
    df['Description'] = df['Description'].fillna("")
    df['Listening Time'] = df['Listening Time'].fillna("Unknown")
    df['Ranks and Genre'] = df['Ranks and Genre'].fillna("Unknown")

    return df

In [30]:
print (df.head)

<bound method NDFrame.head of                                               Book Name                Author  \
0     Think Like a Monk: The Secret of How to Harnes...            Jay Shetty   
1     Ikigai: The Japanese Secret to a Long and Happ...         Héctor García   
2     The Subtle Art of Not Giving a F*ck: A Counter...           Mark Manson   
3     Atomic Habits: An Easy and Proven Way to Build...           James Clear   
4     Life's Amazing Secrets: How to Find Balance an...        Gaur Gopal Das   
...                                                 ...                   ...   
4291  Factfulness: Wie wir lernen, die Welt so zu se...          Hans Rosling   
4292       Late-Talking Children: A Symptom or a Stage?   Stephen M. Camarata   
4293  The Marketing of Evil: How Radicals, Elitists ...        David Kupelian   
4294      Things I Wish I'd Known Before We Got Married          Gary Chapman   
4295  The Disease Delusion: Conquering the Causes of...  Dr. Jeffrey S. Bland  

In [11]:
import pandas as pd
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load data
df = pd.read_csv("cleaned_data.csv")

# -----------------------------
# 🔥 CRITICAL FIX: Fill ALL text columns
# -----------------------------
text_cols = ['Book Name', 'Author', 'Description', 'Ranks and Genre']

for col in text_cols:
    df[col] = df[col].fillna("").astype(str)

# -----------------------------
# Feature Engineering
# -----------------------------
df['content'] = (
    df['Book Name'] + " " +
    df['Author'] + " " +
    df['Description'] + " " +
    df['Ranks and Genre']
)

# Final safety check
df['content'] = df['content'].fillna("")

# -----------------------------
# TF-IDF
# -----------------------------
tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
tfidf_matrix = tfidf.fit_transform(df['content'])

# -----------------------------
# Hybrid Score
# -----------------------------
df['Rating'] = pd.to_numeric(df['Rating'], errors='coerce').fillna(0)
df['norm_rating'] = df['Rating'] / df['Rating'].max()

# Save
pickle.dump(df, open("books.pkl", "wb"))
pickle.dump(tfidf_matrix, open("tfidf_matrix.pkl", "wb"))

print("✅ Model built successfully!")

✅ Model built successfully!


In [26]:
%%writefile app.py
import streamlit as st
import pickle
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import plotly.express as px
import requests
import re

# -----------------------------
# LOAD DATA
# -----------------------------
df = pickle.load(open("books.pkl", "rb"))
tfidf_matrix = pickle.load(open("tfidf_matrix.pkl", "rb"))

st.set_page_config(page_title="📚 Audible Insights", layout="wide")

# -----------------------------
# 🎨 PREMIUM CSS
# -----------------------------
st.markdown("""
<style>
body {background-color: #0e1117; color: white;}
.big-title {
    font-size: 40px;
    font-weight: bold;
    text-align: center;
    background: linear-gradient(to right, #00c6ff, #0072ff);
    -webkit-background-clip: text;
    color: transparent;
}
.card {
    border-radius: 15px;
    padding: 15px;
    background: rgba(255,255,255,0.05);
    backdrop-filter: blur(10px);
    margin-bottom: 20px;
    transition: 0.3s;
}
.card:hover {
    transform: scale(1.03);
    box-shadow: 0px 8px 25px rgba(0,0,0,0.3);
}
.genre-tag {
    display:inline-block;
    padding:4px 10px;
    margin:2px;
    background:#1f77b4;
    border-radius:10px;
    font-size:12px;
}
</style>
""", unsafe_allow_html=True)

# -----------------------------
# 🌟 HEADER
# -----------------------------
st.markdown('<p class="big-title">📚 Audible Insights</p>', unsafe_allow_html=True)
st.markdown("### Discover books tailored to your taste 🚀")

# -----------------------------
# 🧠 CLEAN GENRE FUNCTION
# -----------------------------
def extract_genres(text):
    if pd.isna(text):
        return []

    items = text.split(",")
    clean = []

    for item in items:
        item = item.strip().lower()

        # remove unwanted patterns
        if "star" in item or "%" in item:
            continue

        item = re.sub(r"#\d+", "", item).strip()

        if item != "":
            clean.append(item)

    return clean

# Apply once
df['Cleaned Genres'] = df['Ranks and Genre'].apply(extract_genres)

# Unique genres
all_genres = sorted(set([g for sublist in df['Cleaned Genres'] for g in sublist]))

# -----------------------------
# 📑 TABS
# -----------------------------
tab1, tab2 = st.tabs(["📚 Recommender", "📊 Analytics Dashboard"])

# =========================================================
# 📚 RECOMMENDER
# =========================================================
with tab1:

    st.sidebar.header("🔍 Smart Filters")

    selected_genres = st.sidebar.multiselect("📚 Select Genres", all_genres)
    selected_author = st.sidebar.text_input("👤 Search Author")

    # Filtering
    filtered_df = df.copy()

    if selected_genres:
        filtered_df = filtered_df[
            filtered_df['Cleaned Genres'].apply(
                lambda x: any(g in x for g in selected_genres)
            )
        ]

    if selected_author:
        filtered_df = filtered_df[
            filtered_df['Author'].str.contains(selected_author, case=False, na=False)
        ]

    # Search
    search_query = st.text_input("🔎 Search Book")

    if search_query:
        filtered_df = filtered_df[
            filtered_df['Book Name'].str.contains(search_query, case=False, na=False)
        ]

    # Dropdown
    if len(filtered_df) == 0:
        st.warning("⚠️ No books found")
        st.stop()

    selected_book = st.selectbox("📖 Select Book", filtered_df['Book Name'])

    # -----------------------------
    # HYBRID MODEL
    # -----------------------------
    def hybrid_recommend(book_name, top_n=10):
        idx = df[df['Book Name'] == book_name].index[0]

        similarity = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()
        hybrid_score = 0.7 * similarity + 0.3 * (df['Rating'] / df['Rating'].max())

        df['score'] = hybrid_score

        return df.sort_values(by='score', ascending=False).iloc[1:top_n+1]

    # -----------------------------
    # IMAGE FETCH
    # -----------------------------
    def get_book_image(title):
        try:
            url = f"https://www.googleapis.com/books/v1/volumes?q={title}"
            res = requests.get(url).json()
            return res['items'][0]['volumeInfo']['imageLinks']['thumbnail']
        except:
            return "https://via.placeholder.com/128x180.png?text=No+Image"

    # -----------------------------
    # DISPLAY
    # -----------------------------
    def display_books(results):
        cols = st.columns(3)

        for i, (_, row) in enumerate(results.iterrows()):
            with cols[i % 3]:
                image = get_book_image(row['Book Name'])

                # Genre tags
                tags = "".join([f"<span class='genre-tag'>{g}</span>" for g in row['Cleaned Genres']])

                st.markdown(f"""
                <div class="card">
                    <img src="{image}" width="100%">
                    <h4>📘 {row['Book Name']}</h4>
                    <p><b>👤</b> {row['Author']}</p>
                    <p><b>⭐</b> {round(row['Rating'],2)}</p>
                    {tags}
                </div>
                """, unsafe_allow_html=True)

                st.progress(min(row['Rating']/5, 1.0))

    # Button
    if st.button("✨ Get Smart Recommendations"):
        results = hybrid_recommend(selected_book, 10)
        st.subheader("🔥 Top 10 Picks For You")
        display_books(results)

# =========================================================
# 📊 ANALYTICS DASHBOARD
# =========================================================
with tab2:

    st.header("📊 Data Analytics Dashboard")

    col1, col2, col3 = st.columns(3)
    col1.metric("📚 Total Books", len(df))
    col2.metric("⭐ Avg Rating", round(df['Rating'].mean(), 2))
    col3.metric("👤 Authors", df['Author'].nunique())

    # Rating distribution
    fig1 = px.histogram(df, x="Rating", nbins=20)
    st.plotly_chart(fig1, use_container_width=True)

    # Genre popularity (cleaned)
    genre_series = df['Cleaned Genres'].explode()
    genre_counts = genre_series.value_counts().head(10)

    fig2 = px.bar(x=genre_counts.values, y=genre_counts.index, orientation="h")
    st.plotly_chart(fig2, use_container_width=True)

    # Reviews vs rating
    fig3 = px.scatter(df, x="Number of Reviews", y="Rating", color="Rating")
    st.plotly_chart(fig3, use_container_width=True)

Writing app.py


In [23]:
%%writefile Novella.py

import streamlit as st
import pickle
import plotly.express as px
from sklearn.metrics.pairwise import cosine_similarity
import streamlit as st
import pickle
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import plotly.express as px
import requests

# -----------------------------
# LOAD DATA
# -----------------------------
df = pickle.load(open("books.pkl", "rb"))
tfidf_matrix = pickle.load(open("tfidf_matrix.pkl", "rb"))

st.set_page_config(page_title="📚 Audible Insights", layout="wide")

# -----------------------------
# 🎨 PREMIUM CSS
# -----------------------------
st.markdown("""
<style>
body {
    background-color: #0e1117;
    color: white;
}
.big-title {
    font-size: 40px;
    font-weight: bold;
    text-align: center;
    background: linear-gradient(to right, #00c6ff, #0072ff);
    -webkit-background-clip: text;
    color: transparent;
}
.card {
    border-radius: 15px;
    padding: 20px;
    background: rgba(255,255,255,0.05);
    backdrop-filter: blur(10px);
    margin-bottom: 20px;
    transition: 0.3s;
}
.card:hover {
    transform: scale(1.03);
    box-shadow: 0px 8px 25px rgba(0,0,0,0.3);
}
</style>
""", unsafe_allow_html=True)

# -----------------------------
# 🌟 HEADER
# -----------------------------
st.markdown('<p class="big-title">📚 Audible Insights</p>', unsafe_allow_html=True)
st.markdown("### Discover books tailored to your taste 🚀")

# -----------------------------
# 📑 TABS
# -----------------------------
tab1, tab2 = st.tabs(["📚 Recommender", "📊 Analytics Dashboard"])

# =========================================================
# 📚 TAB 1 → RECOMMENDER SYSTEM
# =========================================================
with tab1:

    st.sidebar.header("🔍 Smart Filters")

    # Extract genres
    genres = set()
    for g in df['Ranks and Genre'].astype(str):
        for item in g.split("|"):
            genres.add(item.strip())

    genre_list = sorted(genres)

    selected_genres = st.sidebar.multiselect("📚 Select Genres", genre_list)
    selected_author = st.sidebar.text_input("👤 Search Author")

    # Apply filters
    filtered_df = df.copy()

    if selected_genres:
        filtered_df = filtered_df[
            filtered_df['Ranks and Genre'].str.contains("|".join(selected_genres), case=False, na=False)
        ]

    if selected_author:
        filtered_df = filtered_df[
            filtered_df['Author'].str.contains(selected_author, case=False, na=False)
        ]

    # Search bar
    search_query = st.text_input("🔎 Search Book")

    if search_query:
        filtered_df = filtered_df[
            filtered_df['Book Name'].str.contains(search_query, case=False, na=False)
        ]

    # Dropdown
    book_list = filtered_df['Book Name'].values

    if len(book_list) == 0:
        st.warning("⚠️ No books found")
        st.stop()

    selected_book = st.selectbox("📖 Select Book", book_list)

    # -----------------------------
    # HYBRID MODEL
    # -----------------------------
    def hybrid_recommend(book_name, top_n=10):
        idx = df[df['Book Name'] == book_name].index[0]

        similarity = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()
        hybrid_score = 0.7 * similarity + 0.3 * (df['Rating'] / df['Rating'].max())

        df['score'] = hybrid_score

        return df.sort_values(by='score', ascending=False).iloc[1:top_n+1]

    # -----------------------------
    # IMAGE FETCH
    # -----------------------------
    def get_book_image(title):
        try:
            url = f"https://www.googleapis.com/books/v1/volumes?q={title}"
            res = requests.get(url).json()
            return res['items'][0]['volumeInfo']['imageLinks']['thumbnail']
        except:
            return "https://via.placeholder.com/128x180.png?text=No+Image"

    # -----------------------------
    # DISPLAY CARDS
    # -----------------------------
    def display_books(results):
        cols = st.columns(3)

        for i, (_, row) in enumerate(results.iterrows()):
            with cols[i % 3]:
                image = get_book_image(row['Book Name'])

                st.markdown(f"""
                <div class="card">
                    <img src="{image}" width="100%">
                    <h4>📘 {row['Book Name']}</h4>
                    <p><b>👤</b> {row['Author']}</p>
                    <p><b>⭐ Rating:</b> {round(row['Rating'],2)}</p>
                    <p><b>🏷</b> {row['Ranks and Genre']}</p>
                </div>
                """, unsafe_allow_html=True)

                st.progress(min(row['Rating']/5, 1.0))

    # -----------------------------
    # BUTTON
    # -----------------------------
    if st.button("✨ Get Smart Recommendations"):
        results = hybrid_recommend(selected_book, 10)

        st.subheader("🔥 Top 10 Picks For You")
        display_books(results)

# =========================================================
# 📊 TAB 2 → ANALYTICS DASHBOARD
# =========================================================
with tab2:

    st.header("📊 Data Analytics Dashboard")

    # KPI Metrics
    col1, col2, col3 = st.columns(3)

    col1.metric("📚 Total Books", len(df))
    col2.metric("⭐ Avg Rating", round(df['Rating'].mean(), 2))
    col3.metric("👤 Unique Authors", df['Author'].nunique())

    # -----------------------------
    # Rating Distribution
    # -----------------------------
    st.subheader("⭐ Rating Distribution")

    fig1 = px.histogram(df, x="Rating", nbins=20)
    st.plotly_chart(fig1, use_container_width=True)

    # -----------------------------
    # Top Authors
    # -----------------------------
    st.subheader("👤 Top Authors")

    top_authors = (
        df.groupby("Author")["Rating"]
        .mean()
        .sort_values(ascending=False)
        .head(10)
        .reset_index()
    )

    fig2 = px.bar(top_authors, x="Rating", y="Author", orientation="h")
    st.plotly_chart(fig2, use_container_width=True)

    # -----------------------------
    # Genre Popularity
    # -----------------------------
    st.subheader("📚 Genre Popularity")

    genre_series = df['Ranks and Genre'].dropna().str.split('|').explode()
    genre_counts = genre_series.value_counts().head(10)

    fig3 = px.bar(x=genre_counts.values, y=genre_counts.index, orientation="h")
    st.plotly_chart(fig3, use_container_width=True)

    # -----------------------------
    # Reviews vs Rating
    # -----------------------------
    st.subheader("🔥 Reviews vs Rating")

    fig4 = px.scatter(df, x="Number of Reviews", y="Rating", color="Rating")
    st.plotly_chart(fig4, use_container_width=True)

    # -----------------------------
    # Price vs Rating
    # -----------------------------
    st.subheader("💰 Price vs Rating")

    fig5 = px.scatter(df, x="Price", y="Rating", color="Rating")
    st.plotly_chart(fig5, use_container_width=True)

Overwriting Novella.py


In [24]:
!streamlit run Novella.py


^C


In [27]:
!streamlit run app.py


^C
